In [9]:
import pandas as pd

df = pd.read_csv('retail_logs.csv')
print("Columns:", df.columns.tolist())
print("\nHead:")
print(df.head())
print("\nInfo:")
print(df.info())

Columns: ['Sale_ID', 'Store_Code', 'Branch', 'Province', 'Region', 'Product_Name', 'Category', 'Sale_Date', 'Quantity', 'Unit_Price', 'Discount_Percent']

Head:
      Sale_ID Store_Code            Branch   Province     Region  \
0  SALE-00264     KKN-01  Khon Kaen Center  Khon Kaen  Northeast   
1  SALE-00077     PKT-01       Phuket Town    Phuket         NaN   
2  SALE-00221     CBI-01          Bangsaen   Chonburi       East   
3  SALE-00150     PKT-01       Phuket Town     Phuket      South   
4  SALE-00084     PKT-01       Phuket Town     PHUKET      South   

    Product_Name     Category    Sale_Date  Quantity  Unit_Price  \
0     Cookie Box       Bakery   2026-03-15         4       150.0   
1     TRAVEL MUG  Merchandise  14-May-2026         4       320.0   
2       Tote Bag  Merchandise  06-May-2026         6       180.0   
3   caesar salad         Food  26-Mar-2026         2       110.0   
4  Mineral Water     Beverage  01-May-2026         6        25.0   

   Discount_Percent  

In [10]:
print("Duplicates in Sale_ID:", df.duplicated("Sale_ID").sum())
print("\nUnique Store_Code to Branch, Province, Region mappings:")
print(df[["Store_Code", "Branch", "Province", "Region"]].drop_duplicates())

print("\nDate Formats sample:")
print(df["Sale_Date"].sample(10, random_state=42))

print("\nNull values count:")
print(df.isnull().sum())

Duplicates in Sale_ID: 5

Unique Store_Code to Branch, Province, Region mappings:
    Store_Code            Branch   Province     Region
0       KKN-01  Khon Kaen Center  Khon Kaen  Northeast
1       PKT-01       Phuket Town    Phuket         NaN
2       CBI-01          Bangsaen   Chonburi       East
3       PKT-01       Phuket Town     Phuket      South
4       PKT-01       Phuket Town     PHUKET      South
..         ...               ...        ...        ...
267     BKK-01    central rama 9   Bangkok    Central 
294     CBI-01          bangsaen   Chonburi       East
321     BKK-02       SIAM SQUARE    Bangkok    CENTRAL
323     BKK-02       Siam Square    BANGKOK   Central 
324     KKN-01  Khon Kaen Center  Khon Kaen  NORTHEAST

[77 rows x 4 columns]

Date Formats sample:
234     04/03/2026
110     2026-06-17
248     27/06/2026
9       17/05/2026
93     15-Jun-2026
219     2026-05-03
287     2026-03-28
198    10-Apr-2026
203     08/06/2026
101     18/04/2026
Name: Sale_Date, dtype:

In [11]:
print("Products and Categories:")
print(df[["Product_Name", "Category"]].drop_duplicates())

print("\nDiscount nulls:")
print(df[df["Discount_Percent"].isnull()])

Products and Categories:
           Product_Name     Category
0            Cookie Box       Bakery
1            TRAVEL MUG  Merchandise
2              Tote Bag  Merchandise
3          caesar salad         Food
4         Mineral Water     Beverage
..                  ...          ...
285      premium coffee     Beverage
292        CAESAR SALAD         Food
312      Thai Milk Tea      Beverage
318          COOKIE BOX       BAKERY
319   Butter Croissant       Bakery 

[69 rows x 2 columns]

Discount nulls:
       Sale_ID Store_Code   Branch  Province Region Product_Name     Category  \
62  SALE-00144     CBI-02  Pattaya  CHONBURI   East   Travel Mug  Merchandise   

      Sale_Date  Quantity  Unit_Price  Discount_Percent  
62  31-May-2026         3       336.0               NaN  


In [12]:
def parse_dates(val):
    for fmt in ["%Y-%m-%d", "%d/%m/%Y", "%d-%b-%Y", "%b %d, %Y", "%d-%m-%Y"]:
        try:
            return pd.to_datetime(val, format=fmt)
        except:
            pass
    return pd.to_datetime(val, errors='coerce')

df['parsed_date'] = df['Sale_Date'].apply(parse_dates)
print("Unparsed dates count:", df['parsed_date'].isnull().sum())
print("Date min/max:", df['parsed_date'].min(), df['parsed_date'].max())

Unparsed dates count: 0
Date min/max: 2026-03-01 00:00:00 2026-06-30 00:00:00


In [13]:
df['Discount_Percent'] = df['Discount_Percent'].fillna(0.0)
df['calculated_total'] = df['Quantity'] * df['Unit_Price'] * (1 - df['Discount_Percent'] / 100.0)
print(df[['Quantity', 'Unit_Price', 'Discount_Percent', 'calculated_total']].head())

   Quantity  Unit_Price  Discount_Percent  calculated_total
0         4       150.0              10.0             540.0
1         4       320.0               5.0            1216.0
2         6       180.0              15.0             918.0
3         2       110.0               0.0             220.0
4         6        25.0              10.0             135.0


In [14]:
print(df.groupby("Store_Code")[["Branch", "Province", "Region"]].agg(lambda x: x.mode()[0]))

                      Branch    Province     Region
Store_Code                                         
BKK-01        Central Rama 9     Bangkok    Central
BKK-02           Siam Square     Bangkok    Central
CBI-01              Bangsaen    Chonburi       East
CBI-02               Pattaya    Chonburi       East
CMI-01                Nimman  Chiang Mai      North
KKN-01      Khon Kaen Center   Khon Kaen  Northeast
PKT-01           Phuket Town      Phuket      South
RYG-01           Rayong City      Rayong       East


In [16]:
import sqlite3
import pandas as pd


def run_etl():
    # ----------------------------------------------------
    # 1. EXTRACT: อ่านข้อมูลจากไฟล์ CSV
    # ----------------------------------------------------
    print("Extracting data...")
    df = pd.read_csv("retail_logs.csv")

    # Rename columns to match expected schema for ETL
    df = df.rename(columns={
        "Sale_ID": "transaction_id",
        "Sale_Date": "transaction_date",
        "Store_Code": "store_id",
        "Branch": "store_name",
        "Province": "city",
        "Product_Name": "product_name",
        "Category": "category",
        "Quantity": "quantity",
        "Unit_Price": "unit_price",
        "Discount_Percent": "discount_percent"
    })

    # Drop duplicate transaction_ids, keeping the first occurrence
    df = df.drop_duplicates(subset=["transaction_id"], keep="first")

    # ----------------------------------------------------
    # 2. TRANSFORM: จัดเตรียมข้อมูลสำหรับแต่ละ Table
    # ----------------------------------------------------
    print("Transforming data...")

    # Convert transaction_date to datetime, coercing errors
    df["transaction_date"] = pd.to_datetime(df["transaction_date"], errors='coerce')
    # Drop rows where transaction_date could not be parsed
    df.dropna(subset=["transaction_date"], inplace=True)


    # 2.1 Dim Date
    dim_date = (
        df[["transaction_date"]]
        .drop_duplicates()
        .copy()
        .reset_index(drop=True)
    )
    dim_date["date_id"] = dim_date["transaction_date"].dt.strftime("%Y%m%d").astype(int)
    dim_date["full_date"] = dim_date["transaction_date"].dt.strftime("%Y-%m-%d")
    dim_date["day"] = dim_date["transaction_date"].dt.day
    dim_date["month"] = dim_date["transaction_date"].dt.month
    dim_date["quarter"] = dim_date["transaction_date"].dt.quarter
    dim_date["year"] = dim_date["transaction_date"].dt.year
    dim_date["day_of_week"] = dim_date["transaction_date"].dt.day_name()
    dim_date = dim_date[
        [
            "date_id",
            "full_date",
            "day",
            "month",
            "quarter",
            "year",
            "day_of_week",
        ]
    ]

    # 2.2 Dim Location
    # Using 'store_id', 'store_name', 'city' directly after renaming
    dim_location = df[["store_id", "store_name", "city"]].drop_duplicates().reset_index(drop=True)

    # 2.3 Dim Product
    # Generate product_id
    dim_product_candidates = df[["product_name", "category"]].drop_duplicates().reset_index(drop=True)
    dim_product_candidates.insert(0, "product_id", range(1, len(dim_product_candidates) + 1))
    dim_product = dim_product_candidates

    # Merge product_id into the main DataFrame for fact_sales
    df = df.merge(dim_product[["product_id", "product_name"]], on="product_name", how="left")

    # 2.4 Fact Sales
    # Ensure discount_percent is filled before calculation. Previous analysis suggests 0.0 for NaN.
    df["discount_percent"] = df["discount_percent"].fillna(0.0)
    # Calculate total_amount considering discount
    df["total_amount"] = df["quantity"] * df["unit_price"] * (1 - df["discount_percent"] / 100.0)
    df["total_amount"] = df["total_amount"].round(2)

    # Create date_id in df for use in fact_sales
    df["date_id"] = df["transaction_date"].dt.strftime("%Y%m%d").astype(int)

    fact_sales = df[
        [
            "transaction_id",
            "date_id",
            "store_id",
            "product_id",
            "quantity",
            "unit_price",
            "total_amount",
        ]
    ].rename(columns={"transaction_id": "sales_id"})

    # ----------------------------------------------------
    # 3. LOAD: เชื่อมต่อและบันทึกลงใน SQLite
    # ----------------------------------------------------
    print("Loading data into SQLite database...")
    conn = sqlite3.connect("retail_warehouse.db")

    # บันทึกข้อมูลลงฐานข้อมูล SQLite
    dim_date.to_sql("dim_date", conn, if_exists="replace", index=False)
    dim_location.to_sql("dim_location", conn, if_exists="replace", index=False)
    dim_product.to_sql("dim_product", conn, if_exists="replace", index=False)
    fact_sales.to_sql("fact_sales", conn, if_exists="replace", index=False)

    conn.close()
    print("ETL Process Completed Successfully!")


if __name__ == "__main__":
    run_etl()

Extracting data...
Transforming data...
Loading data into SQLite database...
ETL Process Completed Successfully!


In [18]:
import sqlite3

query = """
SELECT
    p.product_name,
    p.category,
    SUM(f.quantity) AS total_units_sold,
    ROUND(SUM(f.total_amount), 2) AS total_revenue
FROM fact_sales f
JOIN dim_product p ON f.product_id = p.product_id
GROUP BY p.product_id, p.product_name, p.category
ORDER BY total_revenue DESC
LIMIT 5;
"""

conn = sqlite3.connect("retail_warehouse.db")
df_result = pd.read_sql_query(query, conn)
conn.close()

print(df_result)

       product_name     category  total_units_sold  total_revenue
0        Travel Mug  Merchandise                21        6240.00
1          Tote Bag  MERCHANDISE                29        5152.50
2          Tote Bag  Merchandise                29        5152.50
3  Chicken Sandwich         Food                53        4937.62
4  Chicken Sandwich         FOOD                53        4937.62
